# Classification des Fleurs — CNN (14 espèces)

Notebook complet couvrant les 6 parties de l'exercice :
- **Partie 1** : Exploration et visualisation
- **Partie 2** : Architecture CNN
- **Partie 3** : Optimisation des hyperparamètres
- **Partie 4** : Augmentation des données
- **Partie 5** : Évaluation et analyse
- **Partie 6** : Sauvegarde du modèle

## Installation et imports

In [ ]:
# Installation des dépendances (si nécessaire sous Colab)
!pip install tensorflow matplotlib seaborn scikit-learn numpy pandas -q

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from sklearn.metrics import classification_report, confusion_matrix

print(f'TensorFlow version : {tf.__version__}')
print(f'GPU disponible : {tf.config.list_physical_devices("GPU")}')

In [ ]:
# ── Upload et extraction du ZIP sous Google Colab ────────────────────────
from google.colab import files
import zipfile

uploaded = files.upload()   # Sélectionner Flower_Classification.zip

with zipfile.ZipFile('Flower_Classification.zip', 'r') as z:
    z.extractall('.')

print(' Extraction terminée')
!ls Data/

In [ ]:
# ── Variables globales ───────────────────────────────────────────────────
HEIGHT       = 64           # Résolution réduite pour entraînement rapide
WIDTH        = 64
BATCH_SIZE   = 32
EPOCHS       = 30
NUM_CLASSES  = 14
SEED         = 42

TRAIN_DIR = 'Data/train'
VAL_DIR   = 'Data/val'

CLASS_NAMES = sorted(os.listdir(TRAIN_DIR))
print(f'Classes ({len(CLASS_NAMES)}) : {CLASS_NAMES}')

##  Partie 1 — Exploration et visualisation des données

In [ ]:
# ── Chargement via image_dataset_from_directory ──────────────────────────
train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=(HEIGHT, WIDTH),
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=True,
    seed=SEED
)

val_ds_raw = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=(HEIGHT, WIDTH),
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=False
)

print(f'Classes détectées : {train_ds_raw.class_names}')

In [ ]:
# ── Nombre d'images par classe ───────────────────────────────────────────
print(' Nombre d\'images par classe (train) :')
counts = {}
for cls in CLASS_NAMES:
    n = len(os.listdir(os.path.join(TRAIN_DIR, cls)))
    counts[cls] = n
    print(f'  {cls:<25} : {n}')

print(f'\nTotal train : {sum(counts.values())}')

# Barplot des classes
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(counts.keys(), counts.values(), color='steelblue', edgecolor='white')
ax.set_title('Distribution des images par classe', fontsize=13)
ax.set_xlabel('Espèce de fleur')
ax.set_ylabel('Nombre d\'images')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# ── Fonction : grille 3×3 par classe ─────────────────────────────────────
def visualize_images(class_name, train_dir, n_rows=3, n_cols=3):
    """
    Affiche une grille n_rows×n_cols d'images pour une classe donnée.
    Le nom de la classe apparaît comme titre de la grille.
    """
    class_path = os.path.join(train_dir, class_name)
    images     = os.listdir(class_path)[:n_rows * n_cols]

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(9, 9))
    fig.suptitle(f'Classe : {class_name.replace("_", " ").title()}',
                 fontsize=14, fontweight='bold', y=1.02)

    for idx, ax in enumerate(axes.flatten()):
        if idx < len(images):
            img_path = os.path.join(class_path, images[idx])
            img      = plt.imread(img_path)
            ax.imshow(img)
        ax.axis('off')

    plt.tight_layout()
    plt.show()


# Affichage pour chaque classe
for cls in CLASS_NAMES:
    visualize_images(cls, TRAIN_DIR)

### Analyse — Difficultés prévisibles

| Difficulté | Exemple |
|---|---|
| **Couleurs similaires** | Calendula / California Poppy (orange/jaune) |
| **Formes proches** | Common Daisy / Black-eyed Susan (mêmes pétales rayonnants) |
| **Fond variable** | Herbe, ciel, flou — le fond peut tromper le modèle |
| **Orientations diverses** | Fleurs photographiées de face, de côté, en contre-plongée |
| **Déséquilibre léger** | Astilbe (~726 images) vs Iris (~1041) |
| **Variations intra-classe** | Les roses existent en rouge, blanc, rose, jaune |

Ces difficultés justifient l'utilisation de l'**augmentation de données** (rotation, flip, zoom).

## Partie 2 — Architecture CNN

In [ ]:
# ── Normalisation des pixels [0,255] → [0,1] ─────────────────────────────
normalization_layer = layers.Rescaling(1./255)

train_ds = train_ds_raw.map(lambda x, y: (normalization_layer(x), y))
val_ds   = val_ds_raw.map(lambda x, y: (normalization_layer(x), y))

# Optimisation des performances de chargement
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds   = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
# ── Architecture CNN ──────────────────────────────────────────────────────
# Justification des choix :
#   - 4 blocs Conv + BatchNorm + MaxPool : extraction progressive des features
#   - Filtres croissants (32→64→128→256) : détails simples→complexes
#   - BatchNormalization : stabilise l'entraînement, réduit l'overfitting
#   - Dropout progressif (0.25→0.5) : régularisation
#   - GlobalAveragePooling2D : plus robuste que Flatten sur petites images

def build_cnn(input_shape=(HEIGHT, WIDTH, 3), num_classes=NUM_CLASSES):
    model = models.Sequential([

        # ── Bloc 1 : features bas niveau (bords, textures) ────────────
        layers.Conv2D(32, (3,3), activation='relu', padding='same',
                      input_shape=input_shape),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),

        # ── Bloc 2 : features intermédiaires (formes) ─────────────────
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),

        # ── Bloc 3 : features haut niveau (pétales, structures) ───────
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.3),

        # ── Bloc 4 : features très abstraites ─────────────────────────
        layers.Conv2D(256, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.3),

        # ── Tête de classification ─────────────────────────────────────
        layers.GlobalAveragePooling2D(),
        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(num_classes, activation='softmax')
    ], name='FlowerCNN')

    return model


model = build_cnn()
model.summary()

### Justification architecturale

| Choix | Raison |
|---|---|
| **4 blocs Conv** | Hiérarchie de features : bords → formes → pétales → espèce |
| **Filtres doublés** | 32→64→128→256 : complexité croissante |
| **BatchNorm** | Normalise les activations, accélère la convergence |
| **Dropout 0.25→0.5** | Régularisation progressive, réduit l'overfitting |
| **GlobalAveragePooling** | Réduction spatiale sans aplatissement brutal |
| **Softmax (14 neurones)** | Classification multiclasse (une classe par espèce) |

## Partie 3 — Optimisation des hyperparamètres

In [ ]:
# ── Compilation avec Adam + scheduler ────────────────────────────────────
# Expérimentation :
#   Adam lr=1e-3 → convergence rapide mais instable
#   Adam lr=1e-4 → meilleure généralisation  ← RETENU
#   RMSprop      → similaire à Adam
#   SGD + momentum → plus lent mais parfois meilleur val_acc final

LEARNING_RATE = 1e-4

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ── Callbacks ────────────────────────────────────────────────────────────
callbacks = [
    # Arrêt précoce si val_loss ne s'améliore plus pendant 7 epochs
    EarlyStopping(
        monitor='val_loss',
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),
    # Réduction du taux d'apprentissage si plateau
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
    # Sauvegarde du meilleur modèle
    ModelCheckpoint(
        filepath='best_flower_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

print(' Modèle compilé')
print(f'   Optimiseur : Adam  |  lr={LEARNING_RATE}  |  loss=categorical_crossentropy')

## Partie 4 — Augmentation des données

In [ ]:
# ── ImageDataGenerator avec augmentation ─────────────────────────────────
# Techniques choisies et justification :
#   rotation_range=30    : les fleurs peuvent être photographiées sous n'importe quel angle
#   horizontal_flip      : symétrie naturelle des fleurs
#   zoom_range=0.2       : variations de distance à la fleur
#   width/height_shift   : fleur pas toujours centrée
#   shear_range=0.1      : légère distorsion perspective
#   brightness_range     : conditions d'éclairage variables
#   NE PAS utiliser vertical_flip : les fleurs poussent vers le haut !

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1./255)   # Pas d'augmentation sur val

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(HEIGHT, WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    seed=SEED
)

val_generator = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=(HEIGHT, WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print(' Générateurs créés')

In [ ]:
# ── Visualiser l'effet de l'augmentation ─────────────────────────────────
import matplotlib.image as mpimg
from tensorflow.keras.preprocessing.image import load_img, img_to_array

# Prendre une image de rose comme exemple
sample_path = os.path.join(TRAIN_DIR, 'rose',
                            os.listdir(os.path.join(TRAIN_DIR, 'rose'))[0])
sample_img  = load_img(sample_path, target_size=(HEIGHT, WIDTH))
sample_arr  = img_to_array(sample_img).reshape((1, HEIGHT, WIDTH, 3))

aug_iter = ImageDataGenerator(
    rotation_range=30, horizontal_flip=True,
    zoom_range=0.2, width_shift_range=0.15
).flow(sample_arr, batch_size=1)

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
fig.suptitle('Exemples d\'augmentation sur une image de Rose', fontsize=13)
axes[0][0].imshow(sample_img)
axes[0][0].set_title('Original')
axes[0][0].axis('off')
for i, ax in enumerate(axes.flatten()[1:]):
    aug = next(aug_iter)[0].astype('uint8')
    ax.imshow(aug)
    ax.set_title(f'Aug {i+1}')
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ── Entraînement ─────────────────────────────────────────────────────────
steps_per_epoch  = train_generator.samples // BATCH_SIZE
validation_steps = val_generator.samples   // BATCH_SIZE

history = model.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=EPOCHS,
    validation_data=val_generator,
    validation_steps=validation_steps,
    callbacks=callbacks,
    verbose=1
)

print(' Entraînement terminé')

## Partie 5 — Évaluation et analyse des performances

In [ ]:
# ── Courbes Accuracy & Loss ───────────────────────────────────────────────
def plot_training_history(history):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Courbes d\'entraînement', fontsize=14, fontweight='bold')

    # Accuracy
    axes[0].plot(history.history['accuracy'],     label='Train', color='steelblue')
    axes[0].plot(history.history['val_accuracy'], label='Validation', color='coral')
    axes[0].set_title('Précision (Accuracy)')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Loss
    axes[1].plot(history.history['loss'],     label='Train', color='steelblue')
    axes[1].plot(history.history['val_loss'], label='Validation', color='coral')
    axes[1].set_title('Perte (Loss)')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Analyse
    final_train_acc = history.history['accuracy'][-1]
    final_val_acc   = history.history['val_accuracy'][-1]
    gap = final_train_acc - final_val_acc

    print(f'\nAccuracy finale — Train: {final_train_acc:.3f} | Val: {final_val_acc:.3f}')
    if gap > 0.15:
        print('  Overfitting détecté (gap > 15%) → augmenter dropout ou data augmentation')
    elif final_val_acc < 0.5:
        print('  Underfitting → modèle plus complexe ou plus d\'epochs')
    else:
        print('  Entraînement équilibré')


plot_training_history(history)

In [ ]:
# ── Métriques détaillées : Precision, Recall, F1 ──────────────────────────
val_generator.reset()
y_pred_proba = model.predict(val_generator, verbose=0)
y_pred       = np.argmax(y_pred_proba, axis=1)
y_true       = val_generator.classes

class_labels = list(val_generator.class_indices.keys())

print(' Rapport de classification :')
print(classification_report(y_true, y_pred, target_names=class_labels))

In [ ]:
# ── Matrice de confusion ──────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    cm,
    annot=True, fmt='d',
    cmap='Blues',
    xticklabels=class_labels,
    yticklabels=class_labels,
    linewidths=0.5,
    ax=ax
)
ax.set_title('Matrice de Confusion', fontsize=14, fontweight='bold')
ax.set_xlabel('Prédit')
ax.set_ylabel('Réel')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# ── Visualisation des prédictions (correctes vs erreurs) ──────────────────
def visualize_predictions(model, generator, class_labels, n=16):
    generator.reset()
    images, labels = next(generator)
    preds  = model.predict(images, verbose=0)

    fig, axes = plt.subplots(4, 4, figsize=(14, 14))
    fig.suptitle('Prédictions du modèle (vert=correct, rouge=erreur)', fontsize=13)

    for i, ax in enumerate(axes.flatten()):
        if i >= len(images): break
        true_idx = np.argmax(labels[i])
        pred_idx = np.argmax(preds[i])
        confidence = preds[i][pred_idx]

        color  = 'green' if pred_idx == true_idx else 'red'
        title  = (f'Réel: {class_labels[true_idx]}\n'
                  f'Prédit: {class_labels[pred_idx]}\n'
                  f'Conf: {confidence:.0%}')

        ax.imshow(images[i])
        ax.set_title(title, fontsize=7, color=color)
        for spine in ax.spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(3)
        ax.tick_params(left=False, bottom=False,
                       labelleft=False, labelbottom=False)

    plt.tight_layout()
    plt.show()


val_generator.reset()
visualize_predictions(model, val_generator, class_labels)

In [ ]:
# ── Précision par classe (barplot) ────────────────────────────────────────
per_class_acc = cm.diagonal() / cm.sum(axis=1)

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#2E9E58' if a >= 0.7 else '#D84848' if a < 0.5 else '#EF9F27'
          for a in per_class_acc]
bars = ax.bar(class_labels, per_class_acc, color=colors, edgecolor='white')
ax.axhline(y=0.7, color='gray', linestyle='--', linewidth=1, label='Seuil 70%')
ax.set_title('Accuracy par classe de fleur', fontsize=13, fontweight='bold')
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1.1)
ax.tick_params(axis='x', rotation=45)
ax.legend()
for bar, acc in zip(bars, per_class_acc):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{acc:.0%}', ha='center', fontsize=8)
plt.tight_layout()
plt.show()

print('Classes difficiles (accuracy < 50%) :')
for cls, acc in zip(class_labels, per_class_acc):
    if acc < 0.5:
        print(f'    {cls:<25} : {acc:.0%}')

## Partie 6 — Sauvegarde du modèle

In [ ]:
# ── Sauvegarde ────────────────────────────────────────────────────────────

# Format .h5 (Keras classique)
model.save('flower_cnn_model.h5')
print(' Modèle sauvegardé → flower_cnn_model.h5')

# Format SavedModel (TensorFlow standard)
model.save('flower_cnn_savedmodel')
print(' Modèle sauvegardé → flower_cnn_savedmodel/')

# Téléchargement depuis Colab
from google.colab import files
files.download('flower_cnn_model.h5')

In [ ]:
# ── Rechargement et prédiction sur une nouvelle image ────────────────────
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import load_img, img_to_array

loaded_model = load_model('flower_cnn_model.h5')

def predict_flower(image_path, model, class_labels, img_size=(HEIGHT, WIDTH)):
    """
    Prédit l'espèce d'une fleur à partir d'une image.
    Retourne la classe prédite et sa probabilité.
    """
    img  = load_img(image_path, target_size=img_size)
    arr  = img_to_array(img) / 255.0
    arr  = np.expand_dims(arr, axis=0)

    probs      = model.predict(arr, verbose=0)[0]
    pred_idx   = np.argmax(probs)
    pred_class = class_labels[pred_idx]
    confidence = probs[pred_idx]

    # Affichage
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(img)
    axes[0].set_title(f'Prédiction : {pred_class}\nConfiance : {confidence:.1%}', fontsize=12)
    axes[0].axis('off')

    top5_idx  = np.argsort(probs)[::-1][:5]
    top5_vals = probs[top5_idx]
    top5_labs = [class_labels[i] for i in top5_idx]
    axes[1].barh(top5_labs[::-1], top5_vals[::-1], color='steelblue')
    axes[1].set_xlabel('Probabilité')
    axes[1].set_title('Top 5 prédictions')
    axes[1].set_xlim(0, 1)
    plt.tight_layout()
    plt.show()

    return pred_class, confidence


# Exemple sur la première image de validation
first_val_class = os.listdir(VAL_DIR)[0]
first_val_img   = os.listdir(os.path.join(VAL_DIR, first_val_class))[0]
test_path       = os.path.join(VAL_DIR, first_val_class, first_val_img)

predicted, conf = predict_flower(test_path, loaded_model, class_labels)
print(f'Espèce prédite : {predicted}  ({conf:.1%})')

## Conclusion et pistes d'amélioration

### Résultats obtenus

| Technique | Impact |
|---|---|
| BatchNormalization | Convergence plus rapide et stable |
| Dropout progressif | Réduction de l'overfitting |
| Data Augmentation | Meilleure généralisation |
| ReduceLROnPlateau | Sortie des plateaux d'entraînement |
| EarlyStopping | Évite l'overfitting tardif |

### Pour aller plus loin

1. **Transfer Learning** — utiliser EfficientNetB0 ou MobileNetV2 pré-entraîné sur ImageNet
2. **Résolution plus haute** — passer de 64×64 à 128×128 ou 224×224
3. **Class weights** — compenser le déséquilibre Astilbe (726) vs Iris (1041)
4. **Grad-CAM** — visualiser quelles zones de l'image le modèle regarde

```python
# Transfer Learning (exemple)
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False, weights='imagenet', input_shape=(224, 224, 3)
)
base_model.trainable = False  # Fine-tuning en 2 étapes
```